In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


path = Path(
    "../data/processed/bars_15s/BTCUSDT/"
    "bars_15s_BTCUSDT_2026_06_11.parquet"
)

df = pd.read_parquet(path)

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    utc=True,
)

df = (
    df
    .set_index("timestamp")
    .sort_index()
)

df.head()

,symbol,open,high,low,close,vwap,volume,buy_volume,sell_volume,trade_count
timestamp,,,,,,,,,,
2026-06-11 00:00:00+00:00,BTCUSDT,61511.00,61535.01,61511.00,61529.84,61530.929761,14.08508,1.20852,12.87656,1268
2026-06-11 00:00:15+00:00,BTCUSDT,61530.73,61534.74,61530.72,61534.74,61530.961492,0.35925,0.35560,0.00365,170
2026-06-11 00:00:30+00:00,BTCUSDT,61534.74,61558.00,61534.74,61558.00,61545.700874,1.34527,1.18347,0.16180,890
2026-06-11 00:00:45+00:00,BTCUSDT,61558.00,61574.53,61558.00,61574.52,61566.878010,1.86274,1.82826,0.03448,598
2026-06-11 00:01:00+00:00,BTCUSDT,61574.53,61599.08,61574.52,61592.91,61595.051158,2.36192,1.35038,1.01154,966


In [2]:
day = pd.Timestamp("2026-06-11", tz="UTC")

df_day = df.loc[
    (df.index >= day)
    & (df.index < day + pd.Timedelta(days=1))
].copy()

df_day.head()

,symbol,open,high,low,close,vwap,volume,buy_volume,sell_volume,trade_count
timestamp,,,,,,,,,,
2026-06-11 00:00:00+00:00,BTCUSDT,61511.00,61535.01,61511.00,61529.84,61530.929761,14.08508,1.20852,12.87656,1268
2026-06-11 00:00:15+00:00,BTCUSDT,61530.73,61534.74,61530.72,61534.74,61530.961492,0.35925,0.35560,0.00365,170
2026-06-11 00:00:30+00:00,BTCUSDT,61534.74,61558.00,61534.74,61558.00,61545.700874,1.34527,1.18347,0.16180,890
2026-06-11 00:00:45+00:00,BTCUSDT,61558.00,61574.53,61558.00,61574.52,61566.878010,1.86274,1.82826,0.03448,598
2026-06-11 00:01:00+00:00,BTCUSDT,61574.53,61599.08,61574.52,61592.91,61595.051158,2.36192,1.35038,1.01154,966


In [3]:
print(f"Rows: {len(df_day):,}")
print(f"Start: {df_day.index.min()}")
print(f"End: {df_day.index.max()}")

print()
print(df_day.dtypes)

Rows: 5,760
Start: 2026-06-11 00:00:00+00:00
End: 2026-06-11 23:59:45+00:00

symbol             str
open           float64
high           float64
low            float64
close          float64
vwap           float64
volume         float64
buy_volume     float64
sell_volume    float64
trade_count      int64
dtype: object


In [4]:
invalid_ohlc = df_day.loc[
    (df_day["low"] > df_day["high"])
    | (df_day["open"] < df_day["low"])
    | (df_day["open"] > df_day["high"])
    | (df_day["close"] < df_day["low"])
    | (df_day["close"] > df_day["high"])
]

print(f"Invalid OHLC rows: {len(invalid_ohlc)}")

Invalid OHLC rows: 0


In [5]:
volume_error = (
    df_day["volume"]
    - df_day["buy_volume"]
    - df_day["sell_volume"]
).abs()

print("Maximum volume difference:", volume_error.max())

Maximum volume difference: 1.4210854715202004e-14


In [6]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df_day.index,
        y=df_day["close"],
        mode="lines",
        name="Close",
        line={"width": 1},
    )
)

fig.add_trace(
    go.Scatter(
        x=df_day.index,
        y=df_day["vwap"],
        mode="lines",
        name="VWAP",
        line={"width": 1},
    )
)

fig.update_layout(
    title="BTCUSDT Close and VWAP",
    xaxis_title="Time UTC",
    yaxis_title="Price",
    hovermode="x unified",
    height=600,
)

fig.show()

In [7]:
df_5min = (
    df_day[["buy_volume", "sell_volume"]]
    .resample("15min")
    .sum()
)

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=df_5min.index,
        y=df_5min["buy_volume"],
        name="Buy volume",
    )
)

fig.add_trace(
    go.Bar(
        x=df_5min.index,
        y=df_5min["sell_volume"],
        name="Sell volume",
    )
)

fig.update_layout(
    title="BTCUSDT Buy and Sell Volume",
    xaxis_title="Time UTC",
    yaxis_title="BTC volume",
    barmode="group",
    hovermode="x unified",
    height=600,
)

fig.update_layout(barmode="overlay")
fig.update_traces(opacity=0.50)

fig.show()


In [8]:
total_volume = (
    df_5min["buy_volume"]
    + df_5min["sell_volume"]
)

df_5min["volume_imbalance"] = np.divide(
    df_5min["buy_volume"] - df_5min["sell_volume"],
    total_volume,
    out=np.zeros(len(df_5min), dtype=float),
    where=total_volume.to_numpy() > 0,
)

df_5min[["buy_volume", "sell_volume", "volume_imbalance"]].head()

,buy_volume,sell_volume,volume_imbalance
timestamp,,,
2026-06-11 00:00:00+00:00,68.98604,61.12412,0.060425
2026-06-11 00:15:00+00:00,102.91942,97.38549,0.027628
2026-06-11 00:30:00+00:00,36.83102,33.02545,0.054477
2026-06-11 00:45:00+00:00,87.10078,110.07056,-0.116497
2026-06-11 01:00:00+00:00,270.40177,50.29399,0.686345


In [9]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df_5min.index,
        y=df_5min["volume_imbalance"],
        mode="lines",
        name="Volume imbalance",
        line={"width": 3},
    )
)

fig.add_hline(
    y=0,
    line_width=1,
    line_dash="dash",
)

fig.update_layout(
    title="BTCUSDT Volume Imbalance",
    xaxis_title="Time UTC",
    yaxis_title="Imbalance",
    hovermode="x unified",
    height=500,
)

fig.update_yaxes(
    range=[-1, 1]
)

fig.show()

In [10]:
sample = df_day.loc[
    "2026-06-11 12:00:00":
    "2026-06-11 13:00:00"
].copy()

fig = go.Figure(
    data=[
        go.Candlestick(
            x=sample.index,
            open=sample["open"],
            high=sample["high"],
            low=sample["low"],
            close=sample["close"],
            name="BTCUSDT",
        )
    ]
)

fig.update_layout(
    title="BTCUSDT 15-Second Candlestick Chart",
    xaxis_title="Time UTC",
    yaxis_title="Price",
    height=650,
    xaxis_rangeslider_visible=False,
)

fig.show()

In [11]:
# Moving averages over the 15-second VWAP series
df_day["vwap_ma_5m"] = (
    df_day["vwap"]
    .rolling("5min", min_periods=1)
    .mean()
)

df_day["vwap_ma_15m"] = (
    df_day["vwap"]
    .rolling("15min", min_periods=1)
    .mean()
)


sample = df_day.loc[
    "2026-06-11 12:00:00":
    "2026-06-11 13:00:00"
].copy()


fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    row_heights=[0.7, 0.3],
)

fig.add_trace(
    go.Candlestick(
        x=sample.index,
        open=sample["open"],
        high=sample["high"],
        low=sample["low"],
        close=sample["close"],
        name="BTCUSDT",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=sample.index,
        y=sample["vwap_ma_5m"],
        mode="lines",
        name="VWAP MA 5m",
        line={"width": 1.5},
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=sample.index,
        y=sample["vwap_ma_15m"],
        mode="lines",
        name="VWAP MA 15m",
        line={"width": 1.5},
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Bar(
        x=sample.index,
        y=sample["buy_volume"],
        name="Buy volume",
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Bar(
        x=sample.index,
        y=sample["sell_volume"],
        name="Sell volume",
    ),
    row=2,
    col=1,
)

fig.update_layout(
    title="BTCUSDT Price, VWAP Moving Averages and Aggressive Volume",
    height=800,
    hovermode="x unified",
    barmode="group",
    xaxis_rangeslider_visible=False,
)

fig.update_yaxes(
    title_text="Price",
    row=1,
    col=1,
)

fig.update_yaxes(
    title_text="BTC volume",
    row=2,
    col=1,
)

fig.update_xaxes(
    title_text="Time UTC",
    row=2,
    col=1,
)

fig.show()